# Model registry: which digit classifier should we promote?

Your group serves one model, a small neural network that reads handwritten digits. A teammate's new version reports 1.000 and asks to go live. You need to decide what's in production, say why, and be able to get any released version back a year from now.

A model registry answers two questions: which versions exist, and which one is live. You'll use `gto`, which keeps the registry in Git tags, so there's no server to run. DVC keeps the model files. The model file is cattle, because you can rebuild it. The record of what you released and why is the pet.

**The short path takes about 15 minutes:** run the setup cells, do step 1, then run `python check.py --short`. Steps 2 and 3 are optional and take about 25 more. Only reading? Each step shows the failure on this page and says how to approach the fix.

Cells that start with `# !` are yours. Remove the `#` when you reach them. Everything else runs as it is, once, in order.

## Get a copy to work in

This page is the untouched notebook, run once. It shows the failures and stops, because the fixes are yours. To do the work, use your own copy.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-03-registry?urlpath=lab/tree/registry.ipynb).** It's free and there's nothing to install. The first start can take a few minutes. A Binder session is temporary and has 2 GB of memory, so download anything you want to keep.

**Or run it on your laptop.** This works on Linux and macOS, on x86_64 or ARM, and on Windows through WSL2. You need [Miniforge](https://github.com/conda-forge/miniforge) or any other conda.

```bash
git clone --branch binder-03-registry --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git registry
cd registry
conda env create --file environment.yml
conda activate repro-registry
jupyter lab registry.ipynb
```

On Windows, keep the folder in your WSL2 home, not under `/mnt/c`. To start over at any point, run `python scripts/setup_project.py --reset`, which deletes your work.

## Setup: the first release

The first line makes `!` commands print plain text, so run it again if you restart the kernel. The second builds `project/`, with a pipeline in `dvc.yaml` and a branch from your teammate Sam. It also makes an empty folder, `osn-bucket/`, which stands in for an object storage bucket such as OSN.

In [ ]:
%run scripts/plain_shell.py
!python scripts/setup_project.py

In [ ]:
!cd project && git branch && ls && cat params.yaml dvc.yaml artifacts.yaml

You only run these next cells, but read each one. The width of the network is in `params.yaml`, and `dvc repro` trains it. `split.py` writes the lists of training, validation and test rows to `splits.json`, and Git keeps that file. `train.py` reports accuracy on the validation rows only.

`gto register` gives this commit a version. `gto assign` says which version is in which stage. Both only make Git tags. Look at what `gto show` prints.

In [ ]:
!cd project && dvc repro
!cd project && git add .
!cd project && git commit --quiet -m "Train with 32 hidden units"
!cd project && gto register digits-clf --version v1.0.0
!cd project && gto assign digits-clf --version v1.0.0 --stage prod
!cd project && gto show

`score.py` doesn't use the model in your folder. It asks the registry what's in prod, gets that version, and scores it on the test rows. That's the only place the test rows get used.

In [ ]:
!cd project && python score.py

## Step 1: train another width, and decide

Is another width better? Pick one yourself, and don't assume bigger wins. Train it, register it as a new version, and decide which version goes to prod.

Decide on validation accuracy only. If you pick the model using the test rows, the test score stops being an honest estimate of how it will do on digits it has never seen. So only compare `val_accuracy`.

Your turn. Replace `NEW` with your own width and `CHOSEN` with your decision. If your new width didn't win, skip `gto assign`, because prod is already right. That's a good outcome. Then push.

In [ ]:
# !cd project && python -c "import re; s = open('params.yaml').read(); open('params.yaml', 'w').write(re.sub(r'hidden: \d+', 'hidden: NEW', s))"
# !cd project && dvc repro
# !cd project && dvc metrics diff digits-clf@v1.0.0
# !cd project && git add . && git commit --quiet -m "Train with NEW hidden units"
# !cd project && gto register digits-clf --version v1.1.0
# !cd project && gto assign digits-clf --version CHOSEN --stage prod
# !cd project && dvc push
# !cd project && gto show
# !cd project && gto history digits-clf

### Check the short path

Run this when step 1 is done. `check.py` looks at every version you registered, works out which has the best validation accuracy, then clones your project into an empty folder and rebuilds prod from scratch. Right now it fails, and each line says what to do next. When it passes, your prod version is your best honest model and a stranger can serve it. You can stop here.

In [ ]:
!python check.py --short

## Step 2 (optional): a promotion you should refuse

Sam wants a promotion. Here's the request, and how the candidate compares with your first release.

In [ ]:
!cd project && git show candidate-v2:PROMOTION_REQUEST.md
!cd project && dvc metrics diff digits-clf@v1.0.0 candidate-v2

A classifier that never gets a handwritten digit wrong should make you suspicious. A perfect score can mean the model has seen the answers. The pipeline ran without an error, and the score is real, so no tool will flag this for you. You need a test.

`splits.json` holds three lists of row numbers, under `train`, `val` and `test`. Sam's branch has its own copy.

Your turn:

1. Write `project/tests/test_split.py`. It should read `splits.json` from the project folder and fail if any row is in more than one list.
2. Run it on your branch, where it should pass. Run it on Sam's, where it should fail. The test file isn't committed yet, so it stays in your folder when you switch branches.
3. Write `project/decisions.md`. Refuse the promotion and say what's wrong, so that nobody has to find it again.
4. Commit both.

In [ ]:
# !cd project && python -m pytest -q --color=no tests/test_split.py
# !cd project && git checkout --quiet candidate-v2 && python -m pytest -q --color=no tests/test_split.py
# !cd project && git checkout --quiet main
# !cd project && git add tests decisions.md && git commit --quiet -m "Test that the splits don't overlap, and refuse candidate-v2"

## Step 3 (optional): get an old release back

A year from now, someone asks you to rerun the model behind an old result. This script acts as that person. It clones your project into an empty folder, so it has an empty cache, and tries to restore every registered version by its tag.

In [ ]:
!python scripts/stranger.py

The tag is there and the pointer is there, but the model isn't in the bucket. Nobody ran `dvc push` for that release, and the setup cells above left it out on purpose. A plain `dvc push` only sends what your current files point to, so pushing later doesn't help the earlier version. The model is only in this folder's cache, and when this folder goes, the release goes with it.

Your turn. Get every registered version into the bucket, then run `stranger.py` again until each one restores. The `--help` for `dvc push` will show you more than one way to do it.

In [ ]:
# !cd project && dvc push --help
# !cd project && python ../scripts/stranger.py

## Check everything

This is the full check, for steps 1 to 3. It gives your split test Sam's splits and some bad splits of its own, and it restores each version into an empty folder. It works in throwaway clones, so your project is safe.

In [ ]:
!python check.py

## On your own project

Think of the last model you shared. Could you get that exact file back today, from a clean machine, knowing only its version? Could someone else tell why it was chosen over the others? If not, register it, push it, and write down the decision. Then write the one test that would have caught your worst mistake.

## About the data

The Optical Recognition of Handwritten Digits data: 1,797 images of 8 by 8 pixels. It's included in scikit-learn as `load_digits`, so there's nothing to download. The UCI Machine Learning Repository publishes the data under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), and scikit-learn's code is BSD licensed.

Cite it as: Alpaydin, E. and Kaynak, C. (1998). Optical Recognition of Handwritten Digits. UCI Machine Learning Repository. doi:10.24432/C50P49.